In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')

In [4]:
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [5]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
# Here ,
# Gender and city is the Nominal data so we need oneHot Encoding 
# cough is ordinal data so need Ordinal Encoding

In [7]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [36]:
# fill null values from mean value in column fever
mean = df.mean(numeric_only = True)
df = df.fillna(mean)

In [23]:
# train test split 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['has_covid']), df['has_covid'],test_size = 0.2, random_state = 3)

# AAM ZINDAGI

In [50]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# on test_coloun 
X_test_fever = si.transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [51]:
# ordinalEncoding -> cough
oe = OrdinalEncoder(categories = [['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also on the test data

X_test_cough = oe.transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [52]:
# OneHotEncoding -> Gender, City
ohe = OneHotEncoder(drop = 'first', sparse_output = False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

# also the test data
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

X_train_gender_city.shape

(80, 4)

In [53]:
# Extract Age Columns
X_train_age = X_train.drop(columns = ['gender', 'fever', 'cough', 'city']).values

# for test data
X_test_age = X_test.drop(columns = ['gender', 'fever', 'cough', 'city']).values
X_train_age.shape

(80, 1)

In [56]:
# on train data
X_train_tranformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough),axis = 1)

# also on test data
X_test_tranformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough),axis = 1)

X_train_tranformed

array([[ 34.        ,  98.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   1.        ],
       [ 25.        ,  99.        ,   0.        ,   0.        ,
          1.        ,   0.        ,   1.        ],
       [ 19.        , 100.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   1.        ],
       [ 65.        , 101.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 48.        , 103.        ,   0.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 25.        , 104.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 40.        ,  98.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   1.        ],
       [ 69.        , 102.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 18.        , 104.        ,   0.        ,   0.        ,
          0.    

# Mentos Zindagi

In [ ]:
# column Transform

In [58]:
from sklearn.compose import ColumnTransformer

In [64]:
transformer = ColumnTransformer(transformers = [
    ('tnf1', SimpleImputer(strategy='most_frequent'), ['fever']),
    ('tnf2', OrdinalEncoder(categories = [['Mild', 'Strong']]) ,['cough']),
    ('tnf3', OneHotEncoder(drop = 'first', sparse_output = False), ['gender', 'city'])
    
], remainder = 'passthrough')  # insteat of passthrough we can also drop though columns

In [65]:
transformer

ColumnTransformer(remainder='passthrough',
                  transformers=[('tnf1',
                                 SimpleImputer(strategy='most_frequent'),
                                 ['fever']),
                                ('tnf2',
                                 OrdinalEncoder(categories=[['Mild',
                                                             'Strong']]),
                                 ['cough']),
                                ('tnf3',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['gender', 'city'])])

In [67]:
transformer.fit_transform(X_train)

array([[ 98.        ,   1.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  34.        ],
       [ 99.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  25.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  19.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  65.        ],
       [103.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  48.        ],
       [104.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  25.        ],
       [ 98.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  40.        ],
       [102.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  69.        ],
       [104.        ,   0.        ,   0.        ,   0.        ,
          0.    

In [68]:
transformer.fit_transform(X_train).shape

(80, 7)

In [70]:
transformer.transform(X_test).shape

(20, 7)

# Task of column Transformation

In [55]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder


In [56]:
df = pd.read_csv("student_portuguese_clean.csv")
df.sample(5)

,student_id,school,sex,age,address_type,family_size,parent_status,mother_education,father_education,mother_job,...,family_relationship,free_time,social,weekday_alcohol,weekend_alcohol,health,absences,grade_1,grade_2,final_grade
554,555,MS,F,17,Rural,Greater than 3,Living together,primary education (4th grade),primary education (4th grade),at_home,...,3,5,5,2,2,4,3,10,11,10
297,298,GP,M,17,Rural,Greater than 3,Living together,5th to 9th grade,5th to 9th grade,other,...,5,2,2,1,1,4,0,9,10,10
459,460,MS,F,16,Rural,Greater than 3,Living together,primary education (4th grade),primary education (4th grade),at_home,...,4,4,3,1,1,5,2,10,9,10
409,410,GP,M,18,Urban,Less than or equal to 3,Apart,secondary education,higher education,other,...,4,3,5,1,4,2,9,13,14,15
247,248,GP,M,16,Urban,Greater than 3,Living together,higher education,higher education,health,...,4,2,4,2,4,1,0,13,13,14


In [57]:
df.isnull().sum()

student_id               0
school                   0
sex                      0
age                      0
address_type             0
family_size              0
parent_status            0
mother_education         0
father_education         0
mother_job               0
father_job               0
school_choice_reason     0
guardian                 0
travel_time              0
study_time               0
class_failures           0
school_support           0
family_support           0
extra_paid_classes       0
activities               0
nursery_school           0
higher_ed                0
internet_access          0
romantic_relationship    0
family_relationship      0
free_time                0
social                   0
weekday_alcohol          0
weekend_alcohol          0
health                   0
absences                 0
grade_1                  0
grade_2                  0
final_grade              0
dtype: int64

In [58]:
df.columns

Index(['student_id', 'school', 'sex', 'age', 'address_type', 'family_size',
       'parent_status', 'mother_education', 'father_education', 'mother_job',
       'father_job', 'school_choice_reason', 'guardian', 'travel_time',
       'study_time', 'class_failures', 'school_support', 'family_support',
       'extra_paid_classes', 'activities', 'nursery_school', 'higher_ed',
       'internet_access', 'romantic_relationship', 'family_relationship',
       'free_time', 'social', 'weekday_alcohol', 'weekend_alcohol', 'health',
       'absences', 'grade_1', 'grade_2', 'final_grade'],
      dtype='object')

In [59]:
df = df.drop(columns = ['student_id', 'school', 'address_type',  )

ValueError: Need to specify at least one of 'labels', 'index' or 'columns'

In [53]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns = ['final_grade']),
    df['final_grade'],
    test_size = 0.2,
    random_state = 3)

KeyError: 'final_grade'

In [42]:
df.columns

AttributeError: 'Series' object has no attribute 'columns'

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['final_grade']),
    df['final_grade'],
    test_size=0.2,
    random_state=3
)

KeyError: 'final_grade'